# Shakespeare Says: retrain + export (Colab)

Fine-tunes GPT-2 on cleaned, speaker-formatted Shakespeare, then exports the same ONNX + int8 layout the Next.js app already loads.

1. **Runtime > Change runtime type > T4 GPU**
2. Run the cells top to bottom (about 20-30 min total)
3. At the upload cell, select these 4 files from your repo at once:
   - `data/raw/Shakespeare_data.csv`
   - `scripts/prepare_data.py`
   - `scripts/train_gpt2.py`
   - `scripts/export_onnx.py`
4. Download the zip at the end and unzip it into `src/lib/` (replace the old `shakespeare-gpt2-onnx` folder)

In [ ]:
# 1) GPU check + pinned installs (these versions are known to export a GPT-2 that transformers.js v2 can load)
import sys, subprocess, torch
print('python', sys.version.split()[0], '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch the runtime to T4 GPU first!')

pins = ['transformers==4.46.3', 'accelerate==1.1.1', 'optimum[onnxruntime]==1.23.3', 'onnx==1.17.0', 'onnxruntime==1.20.1']
if sys.version_info < (3, 13):
    pins += ['torch==2.5.1', 'torchvision==0.20.1', 'torchaudio==2.5.1']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pins])
print('installed. (Pip dependency warnings about other Colab packages are harmless.)')

In [ ]:
# 2) Upload the 4 files (Shakespeare_data.csv, prepare_data.py, train_gpt2.py, export_onnx.py)
from google.colab import files
import os, shutil

os.makedirs('data/raw', exist_ok=True)
os.makedirs('scripts', exist_ok=True)
for name in files.upload():
    destination = 'data/raw' if name.endswith('.csv') else 'scripts'
    shutil.move(name, os.path.join(destination, name))
print(sorted(os.listdir('data/raw')), sorted(os.listdir('scripts')))

In [ ]:
# 3) Build + preview the cleaned training samples. They should look like clean 'SPEAKER:' + verse lines, no ACT/SCENE/Enter noise.
!python scripts/prepare_data.py

## Train
About 10-20 min on a T4. Validation loss should drop each epoch, and the generations printed at the very end should read like verse. Tweak with e.g. `!EPOCHS=6 python scripts/train_gpt2.py`.

In [ ]:
# 4) Fine-tune GPT-2 -> build/shakespeare-gpt2
!python scripts/train_gpt2.py

**Stop and read the sample generations above.** If they are still junk, don't export. Try `!EPOCHS=6 python scripts/train_gpt2.py` or tell me what they look like.

In [ ]:
# 5) Export to ONNX, int8-quantize, and compare fp32 vs quantized perplexity -> build/shakespeare-gpt2-onnx(.zip)
!python scripts/export_onnx.py

In [ ]:
# 6) Download (if the browser download stalls on the big file, use the Google Drive cell below instead)
from google.colab import files
files.download('build/shakespeare-gpt2-onnx.zip')

In [ ]:
# Optional: copy the zip (and the PyTorch checkpoint) to Google Drive instead
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/shakespeare-says
!cp build/shakespeare-gpt2-onnx.zip /content/drive/MyDrive/shakespeare-says/
!cd build && zip -qr /content/drive/MyDrive/shakespeare-says/shakespeare-gpt2-pytorch.zip shakespeare-gpt2
!ls -lh /content/drive/MyDrive/shakespeare-says/